# Speed Optimisation Benchmark: `general_magnetar_driven_supernova_diffrax`

Explores three avenues for further speedup beyond the baseline 380× improvement:

| Tier | Technique | Expected gain |
|------|-----------|---------------|
| 1 | Looser ODE tolerances (rtol 1e-5 → 1e-3) | 3–10× per call |
| 2 | `jax.vmap` batched evaluation | throughput × batch size (CPU) |
| 3 | Fixed-step Euler backend | benchmarked below |

**Baseline**: 0.188 ms/call, 380× vs redback 71.3 ms/call.

## 0. Setup & install

In [ ]:
import subprocess, sys, warnings, pathlib
warnings.filterwarnings('ignore')

# Make redback_jax importable when running from notebooks/ subdirectory
_repo_root = str(pathlib.Path().absolute().parent)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

# JAX compatibility shim: jax.tree_leaves / jax.tree_map were removed in JAX
# 0.4.x but the handley-lab blackjax fork still uses them internally.
import jax, jax.tree_util
if not hasattr(jax, 'tree_leaves'):
    jax.tree_leaves = jax.tree_util.tree_leaves
if not hasattr(jax, 'tree_map'):
    jax.tree_map = jax.tree_util.tree_map

# Ensure the handley-lab blackjax fork is installed (provides blackjax.ns.adaptive.nss)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'git+https://github.com/handley-lab/blackjax@proposal', '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip stderr:', result.stderr[-500:])
else:
    print('blackjax fork: OK')

In [ ]:
import os, jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib
matplotlib.rcParams.update({'text.usetex': False})
import matplotlib.pyplot as plt
import time as _time

import bilby
import blackjax
from astropy.cosmology import Planck18
from astropy import constants as cc

from redback.transient_models.supernova_models import (
    general_magnetar_driven_supernova as rb_general_magnetar
)
from redback_jax.models.general_magnetar import (
    general_magnetar_driven_supernova_diffrax,
    general_magnetar_driven_supernova as gm_euler,   # lax.scan fixed-step Euler
)
from redback_jax.inference import (
    Prior, LogUniform, NestedSampler, NSResult, FluxDensityLikelihood
)

# Create output directories
os.makedirs('speed_test_results', exist_ok=True)

print(f'JAX  {jax.__version__}  |  blackjax {blackjax.__version__}  |  bilby {bilby.__version__}')
print(f'Device: {jax.devices()[0]}')

## 1. Synthetic data (SDSS ugriz, 100 observations)

Same dataset as the validation notebook: 20 log-spaced epochs × 5 SDSS bands, 0.05 mag Gaussian noise.

In [ ]:
# ── Truth parameters ──────────────────────────────────────────────────────
TRUTH = dict(
    mej              = 3.5,
    E_sn             = 1.5e51,
    kappa            = 0.1,
    l0               = 1e44,
    tau_sd           = 3e6,
    nn               = 3.0,
    kappa_gamma      = 1.0,
    temperature_floor= 4000.0,
    f_nickel         = 0.0,
    cutoff_wavelength= 3000.0,
)
Z      = 0.05
DL_CM  = float(Planck18.luminosity_distance(Z).cgs.value)
C_CMS  = float(cc.c.cgs.value)

# ── SDSS band centres ──────────────────────────────────────────────────────
BAND_WAV_ANG = np.array([3543., 4770., 6231., 7625., 9134.])
BAND_FREQ_HZ = C_CMS * 1e8 / BAND_WAV_ANG
BANDS  = ['u', 'g', 'r', 'i', 'z']
COLORS = ['violet', 'royalblue', 'green', 'darkorange', 'firebrick']

# Fixed params for JAX model (luminosity_distance is needed explicitly)
FIXED_KW = dict(
    kappa             = TRUTH['kappa'],
    nn                = TRUTH['nn'],
    kappa_gamma       = TRUTH['kappa_gamma'],
    cutoff_wavelength = TRUTH['cutoff_wavelength'],
    f_nickel          = TRUTH['f_nickel'],
    luminosity_distance = DL_CM,
    redshift          = Z,
)

# ── Observation grid ────────────────────────────────────────────────────────
# Interleaved layout: [band0_t0, band1_t0, ..., band4_t0, band0_t1, ...]
N_EPOCHS = 20
T_OBS  = np.geomspace(3., 120., N_EPOCHS)
t_all  = np.repeat(T_OBS, 5)    # shape (100,): t[0..4]=epoch0, t[5..9]=epoch1 ...
nu_all = np.tile(BAND_FREQ_HZ, N_EPOCHS)  # shape (100,): nu[0,5,10,...]=u-band etc.

# ── Generate truth flux: redback called per band (its API takes 1D frequency) ──
_rb_kw = {k: TRUTH[k] for k in ['mej','E_sn','kappa','l0','tau_sd','nn',
                                  'kappa_gamma','temperature_floor',
                                  'cutoff_wavelength','f_nickel']}
F_per_band = []
for nu in BAND_FREQ_HZ:
    F_b = rb_general_magnetar(
        T_OBS, redshift=Z,
        frequency=np.full(N_EPOCHS, nu),
        output_format='flux_density',
        **_rb_kw
    )
    F_per_band.append(F_b)
# Interleave into the same layout as t_all/nu_all
F_true = np.column_stack(F_per_band).ravel()   # shape (100,)

# ── Add 0.05 mag Gaussian noise ─────────────────────────────────────────────
rng      = np.random.default_rng(42)
m_true   = -2.5 * np.log10(F_true / 3.631e6)
m_noisy  = m_true + rng.normal(0, 0.05, size=len(m_true))
F_obs    = 3.631e6 * 10**(-0.4 * m_noisy)
F_err    = F_obs * (np.log(10) / 2.5) * 0.05

print(f'Dataset: {len(F_obs)} observations ({N_EPOCHS} epochs × 5 SDSS bands)')
print(f'Flux range: {F_obs.min():.3e} – {F_obs.max():.3e} mJy')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=False)
# Data layout: t_all = repeat(T_OBS, 5), nu_all = tile(BAND_FREQ_HZ, N_EPOCHS)
# => band b_idx occupies indices b_idx, b_idx+5, b_idx+10, ...
for b_idx, (band, color, ax) in enumerate(zip(BANDS, COLORS, axes)):
    mask = np.arange(b_idx, len(t_all), 5)
    ax.errorbar(t_all[mask], F_obs[mask], F_err[mask],
                fmt='o', color=color, ms=4, lw=0.8, label='noisy data')
    ax.plot(t_all[mask], F_true[mask], '--', color=color, lw=1.5, label='truth')
    ax.set_xlabel('Days (observer frame)')
    ax.set_ylabel('Flux density (mJy)')
    ax.set_title(f'SDSS {band}')
    ax.legend(fontsize=8)
plt.suptitle('Synthetic SDSS ugriz photometry (sigma=0.05 mag noise)', y=1.02)
plt.tight_layout()
os.makedirs('bayesian_results', exist_ok=True)
plt.savefig('bayesian_results/synthetic_data.pdf', bbox_inches='tight')
plt.show()

## 2. redback_jax inference framework

Following `examples/arnett_ns.py` exactly.

### 2.1  Prior

In [ ]:
prior = Prior([
    LogUniform(0.1,   50.,   name='mej'),
    LogUniform(1e49,  1e53,  name='E_sn'),
    LogUniform(1e41,  1e47,  name='l0'),
    LogUniform(1e4,   1e9,   name='tau_sd'),
    LogUniform(1e3,   3e4,   name='temperature_floor'),
])
print(prior)

### 2.2  FluxDensityLikelihood

`FluxDensityLikelihood` follows the same interface as `Likelihood` (both have `_make_log_likelihood(prior)`) but works directly with flux-density models — no `jax_supernovae` dependency.

In [ ]:
likelihood = FluxDensityLikelihood(
    model        = general_magnetar_driven_supernova_diffrax,
    time         = t_all,
    frequency    = nu_all,
    flux_obs     = F_obs,
    flux_err     = F_err,
    fixed_params = FIXED_KW,
)
print(likelihood)

# Build the JIT-compiled log-likelihood once (triggers warmup compilation)
_t0 = _time.perf_counter()
log_like_fn = likelihood._make_log_likelihood(prior)
t_compile = _time.perf_counter() - _t0
print(f'JIT compilation time: {t_compile:.1f} s')

In [ ]:
# ── Sanity check: log-likelihood at truth ──────────────────────────────────
PARAM_NAMES  = ['mej', 'E_sn', 'l0', 'tau_sd', 'temperature_floor']
TRUTH_VALUES = [TRUTH[k] for k in PARAM_NAMES]
theta_truth  = jnp.array(TRUTH_VALUES)

logL_truth = float(log_like_fn(theta_truth))
chi2_expected = len(F_obs) / 2.0
print(f'log L at truth = {logL_truth:.2f}  (expected ~ -{chi2_expected:.0f} for chi^2/dof~1)')

# Steady-state timing (500 calls, drop first 5)
for _ in range(5): log_like_fn(theta_truth).block_until_ready()
times_jax = []
for _ in range(500):
    t0 = _time.perf_counter()
    log_like_fn(theta_truth).block_until_ready()
    times_jax.append(_time.perf_counter() - t0)
t_jax_ms = float(np.median(times_jax)) * 1e3
print(f'JAX log-likelihood:  {t_jax_ms:.3f} ms/call  (median of 500 calls)')

## 3. Redback reference timing

Time the equivalent call using the original redback model to establish a speedup baseline.

In [ ]:
# Redback reference: full 5-band call (same as data generation, per-band loop)
def _rb_full_5band():
    F = []
    for nu in BAND_FREQ_HZ:
        F.append(rb_general_magnetar(
            T_OBS, redshift=Z,
            frequency=np.full(N_EPOCHS, nu),
            output_format='flux_density',
            **_rb_kw
        ))
    return np.column_stack(F).ravel()

# Warmup (3 calls)
for _ in range(3): _rb_full_5band()

times_rb = []
for _ in range(20):
    t0 = _time.perf_counter()
    _rb_full_5band()
    times_rb.append(_time.perf_counter() - t0)
t_rb_ms = float(np.median(times_rb)) * 1e3

speedup = t_rb_ms / t_jax_ms
print(f'redback model call (5 bands):  {t_rb_ms:.1f} ms/call')
print(f'JAX diffrax model (5 bands):   {t_jax_ms:.3f} ms/call')
print(f'Speedup:                        {speedup:.0f}x')

## Tier 1: Loose ODE tolerances

The diffrax Tsit5 solver defaults to `rtol=1e-5, atol=1e-8`. Nested sampling only needs likelihoods accurate enough to *rank* live points, not to match model predictions at the 0.001% level.  Loosening to `rtol=1e-3, atol=1e-6` (100× less strict) should let the adaptive controller take fewer steps.

In [ ]:
# ── Build a loose-tolerance likelihood (pass rtol/atol as fixed params) ──
FIXED_KW_LOOSE = {**FIXED_KW, 'rtol': 1e-3, 'atol': 1e-6}

likelihood_loose = FluxDensityLikelihood(
    model        = general_magnetar_driven_supernova_diffrax,
    time         = t_all,
    frequency    = nu_all,
    flux_obs     = F_obs,
    flux_err     = F_err,
    fixed_params = FIXED_KW_LOOSE,
)

_t0 = _time.perf_counter()
log_like_fn_loose = likelihood_loose._make_log_likelihood(prior)
t_compile_loose = _time.perf_counter() - _t0
print(f'Loose JIT compile: {t_compile_loose:.1f} s')

# Steady-state timing
for _ in range(5): log_like_fn_loose(theta_truth).block_until_ready()
_times_loose = []
for _ in range(500):
    _t0 = _time.perf_counter()
    log_like_fn_loose(theta_truth).block_until_ready()
    _times_loose.append(_time.perf_counter() - _t0)
t_jax_loose_ms = float(np.median(_times_loose)) * 1e3

print(f'\nTight (rtol=1e-5, atol=1e-8): {t_jax_ms:.3f} ms/call')
print(f'Loose (rtol=1e-3, atol=1e-6): {t_jax_loose_ms:.3f} ms/call')
print(f'Tolerance speedup:              {t_jax_ms / t_jax_loose_ms:.1f}x')

In [ ]:
# ── Accuracy check: how much do the outputs change? ─────────────────────
_TRUTH_PARAMS  = {k: TRUTH[k] for k in PARAM_NAMES}
_t_arr  = jnp.array(t_all)
_nu_arr = jnp.array(nu_all)

_F_tight = np.array(general_magnetar_driven_supernova_diffrax(
    _t_arr, _nu_arr, rtol=1e-5, atol=1e-8, **FIXED_KW, **_TRUTH_PARAMS))
_F_loose = np.array(general_magnetar_driven_supernova_diffrax(
    _t_arr, _nu_arr, rtol=1e-3, atol=1e-6, **FIXED_KW, **_TRUTH_PARAMS))

_rel_err    = np.abs(_F_tight - _F_loose) / (_F_tight + 1e-30)
_noise_frac = np.log(10) / 2.5 * 0.05  # 5% mag noise ≈ 11.5% flux noise

print('Relative flux error — loose vs tight — at truth parameters:')
print(f'  Max:    {_rel_err.max():.2e}')
print(f'  Median: {np.median(_rel_err):.2e}')
print(f'  >1%:    {(_rel_err > 0.01).mean():.1%} of data points')
print(f'  >0.1%:  {(_rel_err > 0.001).mean():.1%} of data points')
print()
print(f'Data noise floor: 5% mag ≈ {_noise_frac*100:.1f}% flux noise')
_verdict = 'negligible' if _rel_err.max() < 0.01 * _noise_frac else 'NOT negligible'
print(f'Tolerance error is {_rel_err.max()/_noise_frac*100:.4f}% of 1sigma noise => {_verdict}')

## Tier 2: `jax.vmap` batched evaluation

`jax.vmap(log_like_fn)` vectorises the likelihood over a batch of parameter vectors in a single JIT-compiled kernel call. On CPU this improves memory throughput; on GPU it maps each parameter set to a separate thread-block, turning serial MCMC into massively parallel computation — the key enabler for `blackjax.nss` at `n_live=600`.

In [ ]:
# ── vmap: evaluate N parameter sets simultaneously ───────────────────────
batched_log_like = jax.vmap(log_like_fn_loose)

# Warmup with batch=8
_batch8 = jnp.stack([theta_truth] * 8)
batched_log_like(_batch8).block_until_ready()

batch_sizes = [1, 8, 16, 32, 64, 128, 256, 600]

print(f'{"Batch":>6}  {"Wall ms":>9}  {"ms/call":>9}  '
      f'{"calls/s":>10}  {"vs sequential":>15}')
print('-' * 58)

for bs in batch_sizes:
    _theta_batch = jnp.stack(
        [theta_truth * (1.0 + 0.001 * i) for i in range(bs)]
    )
    # Warmup
    for _ in range(3):
        batched_log_like(_theta_batch).block_until_ready()
    _times = []
    n_reps = max(5, 100 // bs)
    for _ in range(n_reps):
        _t0 = _time.perf_counter()
        batched_log_like(_theta_batch).block_until_ready()
        _times.append(_time.perf_counter() - _t0)
    wall_ms  = float(np.median(_times)) * 1e3
    mpc      = wall_ms / bs
    cps      = bs / wall_ms * 1e3
    spd      = t_jax_loose_ms / mpc
    print(f'{bs:>6}  {wall_ms:>9.2f}  {mpc:>9.3f}  {cps:>10.0f}  {spd:>14.1f}x')

## Tier 3: Fixed-step Euler backend

The model also has a `lax.scan`-based Euler solver (`general_magnetar_driven_supernova`, no `_diffrax` suffix) with a fixed grid of `n_grid` points. Unlike adaptive Tsit5, its per-call cost scales linearly with `n_grid` regardless of the ODE stiffness.  
The question is: what is the minimum `n_grid` that preserves accuracy vs the tight-tolerance diffrax reference?

In [ ]:
# ── Fixed-step Euler vs diffrax Tsit5 ──────────────────────────────────
from redback_jax.models.general_magnetar import (
    general_magnetar_driven_supernova as gm_euler
)

_ALL_PARAMS = {**FIXED_KW, **_TRUTH_PARAMS}  # full param dict for both functions

print(f'{"n_grid":>8}  {"ms/call":>9}  {"vs loose diffrax":>17}  '
      f'{"max rel err vs tight diffrax":>30}')
print('-' * 72)

for n_grid in [128, 256, 512, 1024, 2000]:
    # Warmup (each n_grid is a separate JIT-compiled function)
    for _ in range(3):
        gm_euler(_t_arr, _nu_arr, **_ALL_PARAMS, n_grid=n_grid)

    _times_euler = []
    for _ in range(200):
        _t0 = _time.perf_counter()
        _F = gm_euler(_t_arr, _nu_arr, **_ALL_PARAMS, n_grid=n_grid)
        jax.block_until_ready(_F)
        _times_euler.append(_time.perf_counter() - _t0)
    t_euler_ms = float(np.median(_times_euler)) * 1e3

    _F_euler   = np.array(gm_euler(_t_arr, _nu_arr, **_ALL_PARAMS, n_grid=n_grid))
    _rel_euler = np.abs(_F_euler - _F_tight) / (_F_tight + 1e-30)

    speedup_vs_loose = t_jax_loose_ms / t_euler_ms
    print(f'{n_grid:>8}  {t_euler_ms:>9.3f}  {speedup_vs_loose:>16.1f}x  '
          f'{_rel_euler.max():>30.2e}')

## Summary

In [ ]:
print('=' * 68)
print('  SPEED BENCHMARK  --  general_magnetar_driven_supernova')
print('  100 obs (20 epochs x 5 SDSS ugriz bands)')
print('=' * 68)
print()
print(f'  redback (numpy/scipy):        {t_rb_ms:>8.1f} ms/call  (1x baseline)')
print(f'  JAX diffrax tight (1e-5):     {t_jax_ms:>8.3f} ms/call  '
      f'({t_rb_ms/t_jax_ms:.0f}x faster)')
print(f'  JAX diffrax loose (1e-3):     {t_jax_loose_ms:>8.3f} ms/call  '
      f'({t_rb_ms/t_jax_loose_ms:.0f}x faster, '
      f'{t_jax_ms/t_jax_loose_ms:.1f}x vs tight)')
print()
print(f'  vmap(loose) batch=600 throughput:')
print(f'    CPU: show from table above')
print(f'    GPU (estimated): ~600x throughput from hardware parallelism')
print()
print(f'  Fixed-step Euler: see table above for n_grid accuracy tradeoff')
print()
print('  Recommendation for inference:')
print(f'  Use loose tolerances (rtol=1e-3) -- negligible accuracy loss')
print(f'  vs {_rel_err.max()/_noise_frac*100:.3f}% of 1sigma noise floor')
print('=' * 68)